In [0]:
# =============================================================
# search_utils
# Shared utility functions for vector search and retrieval
# =============================================================

In [0]:
def search_documents(query, source_type, num_results=4, raw=False):
    """
    Queries the vector index for a given source type and returns
    relevant chunks with citations.

    Args:
        query       (str): The search query
        source_type (str): The source domain to search e.g. 'books' or 'docs'
        num_results (int): Number of results to return (default 4)
        raw        (bool): If True, returns plain text content only for agent consumption.
                           If False, returns full data array with metadata (default False)
    
    Returns:
        str  : If raw=True, concatenated chunk content as plain text
        list : If raw=False, raw data array of matching chunks with metadata
    """
    if source_type not in source_config:
        raise ValueError(f"Invalid source_type '{source_type}'. Must be one of: {list(source_config.keys())}")

    index = vsc.get_index(endpoint_name, source_config[source_type]["index"])

    results = index.similarity_search(
        query_text=query,
        columns=["content", "source", "page_number", "start_index"],
        num_results=num_results
    )

    data_array = results.get('result', {}).get('data_array', [])

    if raw:
        return "\n\n".join([row[0] for row in data_array])  # row[0] is the content column
    else:
        return data_array

In [0]:
def run_diagnostic(source_type, query, num_results=4):
    """
    Interactive diagnostic tool for testing vector search results.

    Args:
        source_type (str): The source domain to search e.g. 'books' or 'docs'
        query       (str): The search query to test
        num_results (int): Number of results to return (default 4)
    """
    print(f"\n📡 RAW VECTOR SEARCH DIAGNOSTIC [{source_type.upper()}]: '{query}'")
    print("="*70)

    try:
        docs = search_documents(query, source_type, num_results)

        if not docs:
            print("⚠️ No matches found in the Vector Index.")
        else:
            for i, doc in enumerate(docs):
                content, source, page, start_idx = doc[0], doc[1], doc[2], doc[3]
                print(f"📍 [Match {i+1}] | File: {source} | Page: {page} | Index: {start_idx}")
                print(f"📄 \"{content[:300]}...\"")
                print("-" * 50)

    except Exception as e:
        logger.error(f"❌ Diagnostic failed: {e}")